In [ ]:

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("Libraries imported successfully!")


# ============================================================
# 2. DEVICE
# ============================================================

print("\n================ DEVICE INFORMATION ================")

if torch.cuda.is_available():

    device = torch.device("cuda")

    print("Using GPU")
    print("GPU:", torch.cuda.get_device_name(0))

else:

    device = torch.device("cpu")

    print("Using CPU")


# ============================================================
# 3. LOAD DATASET
# ============================================================

train_file = "train.csv"
test_file = "test.csv"

train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)


# ============================================================
# 4. DATASET INFORMATION
# ============================================================

print("\n================ DATASET INFORMATION ================")

print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)

print("\nTrain Columns:")
print(train_df.columns.tolist())

print("\nTest Columns:")
print(test_df.columns.tolist())

print("\nFirst 5 Training Records:")
display(train_df.head())


# ============================================================
# 5. CLEAN COLUMN NAMES
# ============================================================

train_df.columns = (
    train_df.columns
    .str.strip()
)

test_df.columns = (
    test_df.columns
    .str.strip()
)


# ============================================================
# 6. RENAME COLUMNS
# ============================================================

train_df = train_df.rename(
    columns={
        "Class Index": "label",
        "Title": "title",
        "Description": "description"
    }
)

test_df = test_df.rename(
    columns={
        "Class Index": "label",
        "Title": "title",
        "Description": "description"
    }
)

print("\nColumns after renaming:")
print(train_df.columns.tolist())


# ============================================================
# 7. MISSING VALUES
# ============================================================

print("\n================ MISSING VALUES ================")

print("\nTrain Missing Values:")
print(train_df.isnull().sum())

print("\nTest Missing Values:")
print(test_df.isnull().sum())


# ============================================================
# 8. HANDLE MISSING VALUES
# ============================================================

train_df["title"] = train_df["title"].fillna("")
train_df["description"] = train_df["description"].fillna("")

test_df["title"] = test_df["title"].fillna("")
test_df["description"] = test_df["description"].fillna("")


# ============================================================
# 9. CONVERT LABEL TO NUMERIC
# ============================================================

train_df["label"] = pd.to_numeric(
    train_df["label"],
    errors="coerce"
)

test_df["label"] = pd.to_numeric(
    test_df["label"],
    errors="coerce"
)


# Remove invalid labels

train_df = train_df.dropna(
    subset=["label"]
)

test_df = test_df.dropna(
    subset=["label"]
)


train_df["label"] = (
    train_df["label"]
    .astype(int)
)

test_df["label"] = (
    test_df["label"]
    .astype(int)
)


# ============================================================
# 10. LABEL DISTRIBUTION
# ============================================================

print("\n================ LABEL DISTRIBUTION ================")

print(
    train_df["label"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 11. CONVERT LABELS
# ============================================================
# Original AG News:
#
# 1 = World
# 2 = Sports
# 3 = Business
# 4 = Sci/Tech
#
# DistilBERT:
#
# 0 = World
# 1 = Sports
# 2 = Business
# 3 = Sci/Tech


train_df["label"] = (
    train_df["label"] - 1
)

test_df["label"] = (
    test_df["label"] - 1
)


# ============================================================
# 12. CREATE TEXT COLUMN
# ============================================================

train_df["text"] = (
    train_df["title"].astype(str)
    + " "
    + train_df["description"].astype(str)
)

test_df["text"] = (
    test_df["title"].astype(str)
    + " "
    + test_df["description"].astype(str)
)


# ============================================================
# 13. SELECT REQUIRED COLUMNS
# ============================================================

train_df = train_df[
    ["text", "label"]
].copy()

test_df = test_df[
    ["text", "label"]
].copy()


print("\n================ CLEAN DATA ================")

print("\nFirst 5 Clean Training Records:")

display(train_df.head())

print("\nClean Train Shape:")
print(train_df.shape)

print("\nClean Test Shape:")
print(test_df.shape)


# ============================================================
# 14. CREATE HUGGING FACE DATASETS
# ============================================================

train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df,
    preserve_index=False
)


print("\n================ HUGGING FACE DATASET ================")

print("\nTraining Dataset:")
print(train_dataset)

print("\nTesting Dataset:")
print(test_dataset)


# ============================================================
# 15. LOAD DISTILBERT TOKENIZER
# ============================================================

model_name = "distilbert-base-uncased"

print("\n================ TOKENIZER ================")

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

print("Tokenizer loaded successfully!")


# ============================================================
# 16. TOKENIZATION FUNCTION
# ============================================================

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )


# ============================================================
# 17. TOKENIZE TRAINING DATA
# ============================================================

print("\nTokenizing training data...")

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

print("Training data tokenization completed!")


# ============================================================
# 18. TOKENIZE TEST DATA
# ============================================================

print("\nTokenizing test data...")

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

print("Test data tokenization completed!")


# ============================================================
# 19. REMOVE TEXT COLUMN
# ============================================================

tokenized_train = (
    tokenized_train
    .remove_columns(["text"])
)

tokenized_test = (
    tokenized_test
    .remove_columns(["text"])
)


# ============================================================
# 20. PYTORCH FORMAT
# ============================================================

tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

print("\nTokenized datasets ready!")


# ============================================================
# 21. LABEL INFORMATION
# ============================================================

id2label = {

    0: "World",

    1: "Sports",

    2: "Business",

    3: "Sci/Tech"
}


label2id = {

    "World": 0,

    "Sports": 1,

    "Business": 2,

    "Sci/Tech": 3
}


# ============================================================
# 22. LOAD DISTILBERT MODEL
# ============================================================

print("\n================ MODEL ================")

print("Loading DistilBERT model...")

model = AutoModelForSequenceClassification.from_pretrained(

    model_name,

    num_labels=4,

    id2label=id2label,

    label2id=label2id
)

model.to(device)

print("DistilBERT model loaded successfully!")


# ============================================================
# 23. METRICS
# ============================================================

def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    predictions = np.argmax(
        predictions,
        axis=1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted",
            zero_division=0
        )
    )

    return {

        "accuracy": accuracy,

        "precision": precision,

        "recall": recall,

        "f1": f1
    }


# ============================================================
# 24. TRAINING ARGUMENTS
# ============================================================
#
# IMPORTANT:
# accelerate is required by Trainer.
#
# logging_dir is NOT included because your previous
# Transformers version rejected that parameter.
# ============================================================

training_args = TrainingArguments(

    output_dir="./news_classifier",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=2,

    weight_decay=0.01,

    logging_steps=100,

    load_best_model_at_end=True,

    report_to="none"
)

print("\nTraining arguments created successfully!")


# ============================================================
# 25. CREATE TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_train,

    eval_dataset=tokenized_test,

    processing_class=tokenizer,

    compute_metrics=compute_metrics
)

print("Trainer created successfully!")


# ============================================================
# 26. TRAIN MODEL
# ============================================================

print("\n")
print("============================================================")
print("STARTING DISTILBERT TRAINING")
print("============================================================")

trainer.train()

print("\nTraining completed successfully!")


# ============================================================
# 27. EVALUATE MODEL
# ============================================================

print("\n")
print("============================================================")
print("MODEL EVALUATION")
print("============================================================")

results = trainer.evaluate()

print("\nEvaluation Results:")

for key, value in results.items():

    print(
        key,
        ":",
        value
    )


# ============================================================
# 28. FINAL PERFORMANCE
# ============================================================

print("\n")
print("============================================================")
print("FINAL PERFORMANCE")
print("============================================================")

print(
    "Accuracy:",
    round(
        results["eval_accuracy"] * 100,
        2
    ),
    "%"
)

print(
    "Precision:",
    round(
        results["eval_precision"] * 100,
        2
    ),
    "%"
)

print(
    "Recall:",
    round(
        results["eval_recall"] * 100,
        2
    ),
    "%"
)

print(
    "F1 Score:",
    round(
        results["eval_f1"] * 100,
        2
    ),
    "%"
)


# ============================================================
# 29. TEST PREDICTIONS
# ============================================================

print("\n")
print("============================================================")
print("GENERATING TEST PREDICTIONS")
print("============================================================")

prediction_output = trainer.predict(
    tokenized_test
)

predictions = np.argmax(
    prediction_output.predictions,
    axis=1
)

actual_labels = (
    test_df["label"]
    .values
)


# ============================================================
# 30. CLASSIFICATION REPORT
# ============================================================

print("\n")
print("============================================================")
print("CLASSIFICATION REPORT")
print("============================================================")

print(
    classification_report(

        actual_labels,

        predictions,

        target_names=[

            "World",

            "Sports",

            "Business",

            "Sci/Tech"
        ],

        zero_division=0
    )
)


# ============================================================
# 31. CONFUSION MATRIX
# ============================================================

print("\n")
print("============================================================")
print("CONFUSION MATRIX")
print("============================================================")

cm = confusion_matrix(

    actual_labels,

    predictions
)

print(cm)


# ============================================================
# 32. SAVE MODEL
# ============================================================

print("\n")
print("============================================================")
print("SAVING MODEL")
print("============================================================")

model_path = "./news_classifier_model"

trainer.save_model(
    model_path
)

tokenizer.save_pretrained(
    model_path
)

print(
    "Model saved successfully!"
)

print(
    "Location:",
    model_path
)


# ============================================================
# 33. PREDICTION FUNCTION
# ============================================================

def predict_news(news_text):

    inputs = tokenizer(

        news_text,

        return_tensors="pt",

        truncation=True,

        padding=True,

        max_length=128
    )

    inputs = {

        key: value.to(device)

        for key, value in inputs.items()
    }

    model.eval()

    with torch.no_grad():

        outputs = model(
            **inputs
        )

    probabilities = torch.softmax(

        outputs.logits,

        dim=1
    )

    prediction = torch.argmax(

        probabilities,

        dim=1
    ).item()

    confidence = probabilities[
        0,
        prediction
    ].item()

    category = id2label[
        prediction
    ]

    return category, confidence


# ============================================================
# 34. SAMPLE PREDICTIONS
# ============================================================

print("\n")
print("============================================================")
print("SAMPLE NEWS PREDICTIONS")
print("============================================================")


news1 = """
The stock market rose sharply today after major companies
reported strong quarterly profits.
"""

category, confidence = predict_news(
    news1
)

print("\nNews 1:")
print(news1)

print(
    "Predicted Category:",
    category
)

print(
    "Confidence:",
    round(
        confidence * 100,
        2
    ),
    "%"
)


news2 = """
The football team won the championship after defeating
its opponent in the final match.
"""

category, confidence = predict_news(
    news2
)

print("\nNews 2:")
print(news2)

print(
    "Predicted Category:",
    category
)

print(
    "Confidence:",
    round(
        confidence * 100,
        2
    ),
    "%"
)


news3 = """
Scientists developed a new computer technology that can
improve artificial intelligence systems.
"""

category, confidence = predict_news(
    news3
)

print("\nNews 3:")
print(news3)

print(
    "Predicted Category:",
    category
)

print(
    "Confidence:",
    round(
        confidence * 100,
        2
    ),
    "%"
)


news4 = """
World leaders met to discuss international relations
and global security.
"""

category, confidence = predict_news(
    news4
)

print("\nNews 4:")
print(news4)

print(
    "Predicted Category:",
    category
)

print(
    "Confidence:",
    round(
        confidence * 100,
        2
    ),
    "%"
)


# ============================================================
# 35. CUSTOM NEWS CLASSIFIER
# ============================================================

print("\n")
print("============================================================")
print("CUSTOM NEWS CLASSIFIER")
print("============================================================")

print("Enter your own news article.")
print("Type 'exit' to stop.")


while True:

    user_news = input(
        "\nEnter news text: "
    )

    if user_news.lower() == "exit":

        print("\nProgram completed.")

        break

    if user_news.strip() == "":

        print(
            "Please enter some news text."
        )

        continue

    category, confidence = (
        predict_news(user_news)
    )

    print(
        "\nPredicted Category:",
        category
    )

    print(
        "Confidence:",
        round(
            confidence * 100,
            2
        ),
        "%"
    )